# Assignment 2: A Short Data Story

## Formalia

**Name:** Frederik Sunesen  
**Due date and time:** Monday April 6th, 2026 at 23:55  
**Link to Website:** [The Pulse of San Francisco](https://DrSune.github.io/socialdata2026/assignment2.html)

## Project Overview
This assignment follows the **Magazine Genre** from the Segel & Heer paper. It explores the relationship between "Opportunity Crimes" (like Larceny) and "Enforcement Crimes" (like Drug Offenses) in San Francisco, utilizing data from 2003 to 2025.

---

## 1. Data Processing and Visualization Generation

The following code blocks replicate the generation of the three core visualizations used in the data story.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import plotly.express as px
import plotly.graph_objects as go
import os

# Ensure docs directory exists
if not os.path.exists('../docs'):
    os.makedirs('../docs')

# Load data
df_hist = pd.read_csv('../solutions/hist.csv')
df_recent = pd.read_csv('../solutions/recent.csv')

print("Data loaded successfully.")

### Visualization 1: Static Hourly Rhythm (SVG)
This chart highlights the different "heartbeats" of Larceny vs. Drug Offenses.

In [ ]:
# Prep Data
h = df_hist[['Category', 'Time']].copy()
r = df_recent[['Incident Category', 'Incident Time']].copy()
r.columns = h.columns
df_time = pd.concat([h, r], ignore_index=True).dropna()
df_time['Time_dt'] = pd.to_datetime(df_time['Time'], format='mixed')
df_time['Hour'] = df_time['Time_dt'].dt.hour

focus = ['LARCENY THEFT', 'BURGLARY', 'DRUG/NARCOTIC']
hourly = df_time[df_time['Category'].isin(focus)].groupby(['Hour', 'Category']).size().unstack()
hourly_norm = hourly / hourly.sum()

# Plot
plt.figure(figsize=(12, 6))
for cat in focus:
    plt.plot(hourly_norm.index, hourly_norm[cat], label=cat, linewidth=3, marker='o', markersize=4)

plt.title('The Daily Rhythm of SF Crime: Opportunity vs. Enforcement', fontsize=16)
plt.xlabel('Hour of Day (24h)')
plt.ylabel('Proportion of Daily Total')
plt.legend(title='Crime Type')
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig('../docs/hourly_rhythm.svg', bbox_inches='tight')
plt.show()

### Visualization 2: Geographic Concentration Map (HTML)
A heatmap showing the extreme concentration of drug enforcement in the Tenderloin.

In [ ]:
drugs = df_recent[df_recent['Incident Category'] == 'Drug Narcotic'].dropna(subset=['Latitude', 'Longitude'])
m = folium.Map(location=[37.7749, -122.4194], zoom_start=13, tiles='CartoDB positron')
HeatMap([[row['Latitude'], row['Longitude']] for _, row in drugs.iterrows()], radius=10, blur=15).add_to(m)
folium.Marker([37.7842, -122.4140], popup='Tenderloin Hub').add_to(m)
m.save('../docs/drug_heatmap.html')
m

### Visualization 3: Interactive Trends (Plotly HTML)
An interactive look at the decoupling of trends during the pandemic.

In [ ]:
df_hist['Year'] = pd.to_datetime(df_hist['Date'], format='mixed').dt.year
df_recent['Year'] = pd.to_datetime(df_recent['Incident Date'], format='mixed').dt.year

yearly_h = df_hist[df_hist['Category'].isin(['LARCENY/THEFT', 'DRUG/NARCOTIC'])].groupby(['Year', 'Category']).size().reset_index(name='Counts')
yearly_r = df_recent[df_recent['Incident Category'].isin(['Larceny Theft', 'Drug Narcotic'])].groupby(['Year', 'Incident Category']).size().reset_index(name='Counts')
# (Simplified grouping for demonstration)
fig = px.line(yearly_h, x='Year', y='Counts', color='Category', title='Long-term Trends')
fig.write_html('../docs/interactive_trends.html', include_plotlyjs='cdn')
fig.show()

---

## 2. Exercise 3.1: Reflection

**1. What was the hardest part of creating the data story — the analysis, the visualization, the writing, or the web design? Why?**
> The hardest part was the **writing and narrative structure**. While generating charts is a technical skill, deciding *what* to say and how to frame it as a coherent argument (without just listing facts) requires a different kind of critical thinking. Specifically, striking the right balance between being objective and acknowledging the systemic biases (like enforcement vs. actual crime) was a challenge.

**2. Think about the author-driven to reader-driven spectrum one more time. Your data story is heavily author-driven (you chose what to show, in what order). But your interactive Plotly chart gives the reader some freedom to explore. Is that tension a problem, or does it work? How does the interactivity change the reading experience compared to the static figures?**
> The tension actually works as a **"Martini Glass"** structure. The static figures and text guide the reader through the core argument (the stem of the glass), and the interactive Plotly chart at the end allows them to "verify" the findings or explore specific sub-trends (the flare of the glass). It transitions the reader from being a passive consumer of information to an active participant, which increases trust in the analysis.

**3. If you had unlimited time, what would you add or change about your story?**
> I would add **scrollytelling** elements where the map updates or zooms automatically as the reader scrolls through specific paragraphs about different neighborhoods. I would also integrate more qualitative data, such as news headlines from specific dates when spikes occurred, to provide even more context for the "why" behind the numbers.